## To describe the changes in Singapore's population by planning area

Region boundary is obtained from data.gov.sg

2008 and 2014 files are in SHP format, so they will be changes to gpkg for standardisation. 

`ogr2ogr -f GPKG region_boundary_2008.gpkg MasterPlan2008RegionBoundaryNoSeaSHP -nln region_boundary_2008 -nlt MULTIPOLYGON`

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import shape
from pathlib import Path
import os
from bs4 import BeautifulSoup
import re

In [2]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

BASE_DATASET_PATH = Path(SRC_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

MISSING_VALUES = ["nan", "-", "NaN", 'N/A', 'NA', "na"]

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


In [3]:
region_boundary_2019_filepath = Path(BASE_DATASET_PATH / "singapore_data/onemap_data/region_boundary_2019.gpkg")
region_boundary_2019 = gpd.read_file(region_boundary_2019_filepath)

print(region_boundary_2019["Description"][1])
region_boundary_2019.columns

<center><table><tr><th colspan='2' align='center'><em>Attributes</em></th></tr><tr bgcolor="#E3E3F3"> <th>REGION_N</th> <td>NORTH REGION</td> </tr><tr bgcolor=""> <th>REGION_C</th> <td>NR</td> </tr><tr bgcolor="#E3E3F3"> <th>INC_CRC</th> <td>9642FA9C45608EBD</td> </tr><tr bgcolor=""> <th>FMEL_UPD_D</th> <td>20191223152214</td> </tr></table></center>


Index(['Name', 'Description', 'REGION_N', 'geometry'], dtype='object')

In [4]:
# extract out the region name from the Description column
def extract_region_name(description_html):
    """
    Parses the HTML-like string in the Description column to extract 
    the value associated with the 'REGION_N' key.
    
    Args:
        description_html (str): The string content from the 'Description' column.
        
    Returns:
        str: The extracted region name (e.g., 'NORTH REGION') or None if not found.
    """
    if pd.isna(description_html):
        return None
        
    try:
        # Use BeautifulSoup to parse the table structure
        soup = BeautifulSoup(description_html, 'html.parser')
        
        # Find all table rows (<tr>)
        rows = soup.find_all('tr')
        
        # Iterate through the rows to find the one containing 'REGION_N'
        for row in rows:
            # Look for the cell containing the attribute key 'REGION_N'
            # Note: The structure is <th>REGION_N</th><td>NORTH REGION</td>
            if row.find('th', string='REGION_N'):
                # The value is typically in the next cell (<td>)
                # We target the sibling <td> which holds the value
                region_name_tag = row.find('td')
                if region_name_tag:
                    return region_name_tag.text.strip()
                    
        return None # Return None if the key 'REGION_N' is not found
        
    except Exception as e:
        print(f"Error parsing description for a feature: {e}")
        return None

In [5]:
def clean_subzone_column(df, column_name):
    """
    Cleans the specified column of the DataFrame based on the provided logic:
    1. Converts all values to string type.
    2. Removes leading/trailing whitespace.
    3. Filters out rows where the resulting string is "nan".

    Parameters:
        df (pd.DataFrame): The input DataFrame.
        column_name (str): The name of the column to clean.

    Returns:
        pd.DataFrame: The DataFrame with the column cleaned and filtered.
    """
    # 1. Convert all values in the column to string type
    df[column_name] = df[column_name].astype(str)

    # 2. Remove leading and trailing whitespace and set them to lowercase
    df[column_name] = df[column_name].str.strip().str.lower()

    # 3. Filter out rows where the value is the string "nan" (case-sensitive)
    # The filtering *removes* rows where the condition is True
    df = df[df[column_name] != "nan"]

    return df

In [6]:
from collections import Counter

def find_all_duplicates(data_list):
    """Returns a list of elements that appear more than once."""
    counts = Counter(data_list)
    
    # Filter the counter to find items with a count > 1
    duplicates = [item for item, count in counts.items() if count > 1]
    
    return duplicates

In [7]:
def convert_cols_to_int(df):
    """
    convert the values of columns other than planning_area and subzone into int. 
    """

    exclude_cols = ['planning_area', 'subzone']

    columns_to_convert = [col for col in df.columns if col not in exclude_cols]

    # Clean and Convert the columns
    for col in columns_to_convert:
        # A. Fill any remaining true NaN values with 0. 
        #    This is necessary because standard int types cannot store NaN.
        #    You may need to convert the column to numeric first to ensure 
        #    any mixed types or lingering strings are handled.
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
        
        # B. Cast the cleaned (now float) column to the integer type
        #    Using .astype('Int64') is often preferred over 'int64' if you want 
        #    to retain the ability to use Pandas' nullable integer type, but 
        #    if you filled NaNs with 0, 'int64' is sufficient.
        df[col] = df[col].astype('int64')

In [8]:
# create a new column with the region names extracted out from each row
region_boundary_2019["REGION_N"] = region_boundary_2019["Description"].apply(extract_region_name)

save_to_filepath = Path(BASE_DATASET_PATH / "singapore_data/onemap_data/region_boundary_2019.gpkg")

# Overwrite the layer in the GeoPackage file
# CRITICAL: mode='w' will overwrite the existing layer with the same name.
# We must ensure to use the same gpkg_file and layer name.
region_boundary_2019.to_file(
    save_to_filepath, 
    layer="region_boundary_2019", 
    driver='GPKG', 
    mode='w'  # Use 'w' (write mode) to overwrite the existing layer
)

region_boundary_2019

,Name,Description,REGION_N,geometry
0,kml_1,<center><table><tr><th colspan='2' align='cent...,WEST REGION,"MULTIPOLYGON Z (((103.74134 1.15997 0, 103.741..."
1,kml_2,<center><table><tr><th colspan='2' align='cent...,NORTH REGION,"MULTIPOLYGON Z (((103.87242 1.43995 0, 103.872..."
2,kml_3,<center><table><tr><th colspan='2' align='cent...,NORTH-EAST REGION,"MULTIPOLYGON Z (((103.94702 1.40759 0, 103.947..."
3,kml_4,<center><table><tr><th colspan='2' align='cent...,EAST REGION,"MULTIPOLYGON Z (((104.07166 1.29195 0, 104.071..."
4,kml_5,<center><table><tr><th colspan='2' align='cent...,CENTRAL REGION,"MULTIPOLYGON Z (((103.83604 1.21274 0, 103.835..."


In [9]:
population_2015_filepath = Path(BASE_DATASET_PATH / "singapore_data/data_gov/2015_age_group_subzone_edited.xlsx")
population_2020_filepath = Path(BASE_DATASET_PATH / "singapore_data/singstat/2020_age_group_subzone_shorten.xlsx")

population_2015_df = pd.read_excel(population_2015_filepath, usecols = lambda column: not column.startswith('Unnamed:'))
population_2020_df = pd.read_excel(population_2020_filepath, sheet_name = "planning_area")

# make the column names all lower case
population_2015_df.columns = [x.lower() for x in population_2015_df.columns]
population_2020_df.columns = [x.lower() for x in population_2020_df.columns]

# convert all elements in the dataframe to string and lower case
population_2015_df = population_2015_df.apply(lambda x: x.astype(str).str.lower())
population_2020_df = population_2020_df.apply(lambda x: x.astype(str).str.lower())
# removing "\n"
population_2015_df['planning_area'] = population_2015_df['planning_area'].str.replace(r'\n', '', regex=True)
population_2020_df['planning_area'] = population_2020_df['planning_area'].str.replace(r'\n', '', regex=True)

population_2020_df.shape

(56, 21)

In [10]:
# Condition A: planning_area contains the string "- total"
mask_dash_total = population_2015_df["planning_area"].str.contains("- total", na=False)

# Condition B: planning_area contains the exact string "total"
# Use .eq() for an exact match
mask_exact_total = population_2015_df["planning_area"].eq("total")

# Combine the masks using the logical OR operator (|)
mask = mask_dash_total | mask_exact_total

planning_area_2015 = population_2015_df[mask].copy()
planning_area_2015['planning_area'] = planning_area_2015['planning_area'].str.replace('- total', '').str.strip()
print(planning_area_2015.shape)
planning_area_2015.head()

(56, 59)


,planning_area,subzone,total_total,total_0_4,total_5_9,total_10_14,total_15_19,total_20_24,total_25_29,total_30_34,...,females_40_44,females_45_49,females_50_54,females_55_59,females_60_64,females_65_69,females_70_74,females_75_79,females_80_84,females_85andover
0,total,total,3902690,183580,204450,214390,242900,264130,271030,290620,...,162300,153810,156630,147200,120830,93730,54850,45090,30850,27660
1,ang mo kio,total,174770,6790,7660,8290,9320,10310,11170,12250,...,7100,6660,7010,7120,6940,5810,3600,2800,1910,1520
14,bedok,total,289750,11690,13400,14750,16930,19450,19860,19270,...,11660,10860,11870,11880,10570,8570,4880,4060,2840,2700
23,bishan,total,90700,3430,4330,4710,5520,6860,6460,5720,...,3750,3590,4070,3960,3230,2520,1460,1180,840,770
27,boon lay,total,30,na,na,na,na,na,na,na,...,na,na,na,na,na,na,na,na,na,na


In [11]:
areas_2015 = planning_area_2015["planning_area"].tolist()
areas_2015 = [s.replace('- total', '') for s in areas_2015]
areas_2015.sort()
areas_2020 = population_2020_df["planning_area"].tolist()
areas_2020 = [s.replace('\n', '') for s in areas_2020]
index_of_total = areas_2020.index("total")
del areas_2020[index_of_total]
areas_2020.sort()
print("Number of planning area listed in 2015 data:", len(areas_2015))
print(areas_2015)
print("Number of planning area listed in 2020 data:", len(areas_2020))
print(areas_2020)


Number of planning area listed in 2015 data: 56
['ang mo kio', 'bedok', 'bishan', 'boon lay', 'bukit batok', 'bukit merah', 'bukit panjang', 'bukit timah', 'central water catchment', 'changi', 'changi bay', 'choa chu kang', 'clementi', 'downtown core', 'geylang', 'hougang', 'jurong east', 'jurong west', 'kallang', 'lim chu kang', 'mandai', 'marina east', 'marina south', 'marine parade', 'museum', 'newton', 'north-eastern islands', 'novena', 'orchard', 'outram', 'pasir ris', 'paya lebar', 'pioneer', 'punggol', 'queenstown', 'river valley', 'rochor', 'seletar', 'sembawang', 'sengkang', 'serangoon', 'simpang', 'singapore river', 'southern islands', 'straits view', 'sungei kadut', 'tampines', 'tanglin', 'tengah', 'toa payoh', 'total', 'tuas', 'western islands', 'western water catchment', 'woodlands', 'yishun']
Number of planning area listed in 2020 data: 55
['ang mo kio', 'bedok', 'bishan', 'boon lay', 'bukit batok', 'bukit merah', 'bukit panjang', 'bukit timah', 'central water catchment',

In [12]:
# if the planning areas have the same name and non of them are missing, an empty set will be returned.
missing_areas_of_areas_2015 = set(areas_2020).difference(set(areas_2015))
print(missing_areas_of_areas_2015)

missing_areas_of_areas_2020 = set(areas_2015).difference(set(areas_2020))
print(missing_areas_of_areas_2020)

{'southernislands'}
{'total', 'southern islands'}


### Southern Islands are named differently for both datasets
Checking the planning_areas obtained from onemap, Southern Islands contains spacing

Elements containing {Southern Islands, SouthernIslands} will be mapped to "Southern Islands" 

In [13]:
# The mapping dictionary
area_name_mapping = {
    'southernislands': 'southern islands',
    'Southern Islands': 'southern islands'
}

population_2020_df['planning_area'] = population_2020_df['planning_area'].map(area_name_mapping).fillna(population_2020_df['planning_area'])
# population_2020_df


In [14]:
# Handle String Representations: Replace all strings that look like missing values
population_2020_df = (
    population_2020_df.replace(MISSING_VALUES, np.nan, regex=False))

planning_area_2015 = (
    planning_area_2015.replace(MISSING_VALUES, np.nan, regex=False))

convert_cols_to_int(population_2020_df)
convert_cols_to_int(planning_area_2015)

#### Sum up population that is over 60 by planning area

In [15]:
cols_2015 = ['total_60_64', 'total_65_69', 'total_70_74', 'total_75_79', 'total_80_84', 'total_85andover']
cols_2020 = ['60 - 64', '65 - 69', '70 - 74', '75 - 79', '80 - 84', '85 - 89', '90 & over']

planning_area_2015["total_above_60"] = planning_area_2015[cols_2015].sum(axis=1)
population_2020_df["total_above_60"] = population_2020_df[cols_2020].sum(axis=1)

In [16]:
def save_to_excel_with_multiple_sheets(df, filepath, sheet_name_to_update= None):
    if filepath.exists():
        excel_mode = 'a'
        writer_kwargs = {'mode': excel_mode, 'if_sheet_exists': 'replace'}
    else:
        excel_mode = 'w'
        writer_kwargs = {'mode': excel_mode}

    with pd.ExcelWriter(
        filepath, 
        engine='openpyxl',
        **writer_kwargs # Unpack the dictionary of arguments
    ) as writer:
        # Write the DataFrame to the specified sheet
        df.to_excel(
            writer, 
            sheet_name=sheet_name_to_update, 
            index=False # Set to True if you want to save the DataFrame index
        )
    print(f"Save to filepath: {filepath}" )

#### Save the cleaned planning area data

In [17]:
population_2015_filepath = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/2015_age_group_planning_area_subzone.xlsx")
population_2020_filepath = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/2020_age_group_planning_area_subzone.xlsx")

save_to_excel_with_multiple_sheets(population_2020_df, population_2020_filepath, "planning_area")

save_to_excel_with_multiple_sheets(planning_area_2015, population_2015_filepath, "planning_area")

Save to filepath: /Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/singapore_data/cleaned_data/2020_age_group_planning_area_subzone.xlsx
Save to filepath: /Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/singapore_data/cleaned_data/2015_age_group_planning_area_subzone.xlsx


### Population dataset now contains planning areas for both 2015 and 2020. 

In [18]:
# population_2015_filepath = Path(BASE_DATASET_PATH / "singapore_data/data_gov/2015ResidentPopulationbyPlanningAreaSubzoneAgeGroup.csv")
population_2020_filepath = Path(BASE_DATASET_PATH / "singapore_data/singstat/2020_age_group_subzone_shorten.xlsx")

# population_by_subzone_2015 = pd.read_csv(population_2015_filepath)
population_by_subzone_2020 = pd.read_excel(population_2020_filepath, sheet_name = "subzone")

# make the column names all lower case
# population_by_subzone_2015.columns = [x.lower() for x in population_by_subzone_2015.columns]
population_by_subzone_2020.columns = [x.lower() for x in population_by_subzone_2020.columns]

# convert all elements in the dataframe to string and lower case
# population_by_subzone_2015 = population_by_subzone_2015.apply(lambda x: x.astype(str).str.lower())
population_by_subzone_2020 = population_by_subzone_2020.apply(lambda x: x.astype(str).str.lower())

# removing "\n"
# population_by_subzone_2015['number'] = population_by_subzone_2015['number'].str.replace(r'\n', '', regex=True)
population_by_subzone_2020['planning_area'] = population_by_subzone_2020['planning_area'].str.replace(r'\n', '', regex=True)
population_by_subzone_2020['subzone'] = population_by_subzone_2020['subzone'].str.replace(r'\n', '', regex=True)

# use the partially cleaned 2015 data from above
population_by_subzone_2015 = population_2015_df.copy()


In [19]:
population_by_subzone_2015.head()

,planning_area,subzone,total_total,total_0_4,total_5_9,total_10_14,total_15_19,total_20_24,total_25_29,total_30_34,...,females_40_44,females_45_49,females_50_54,females_55_59,females_60_64,females_65_69,females_70_74,females_75_79,females_80_84,females_85andover
0,total,total,3902690,183580,204450,214390,242900,264130,271030,290620,...,162300,153810,156630,147200,120830,93730,54850,45090,30850,27660
1,ang mo kio- total,total,174770,6790,7660,8290,9320,10310,11170,12250,...,7100,6660,7010,7120,6940,5810,3600,2800,1910,1520
2,nan,ang mo kio town centre,5020,260,280,320,280,260,310,370,...,260,210,170,170,160,150,80,70,40,40
3,nan,cheng san,29770,1290,1180,1290,1400,1570,1830,2490,...,1260,1110,1140,1220,1270,970,580,420,280,260
4,nan,chong boon,27900,910,1100,1180,1370,1520,1800,1980,...,1060,1080,1120,1160,1200,1050,690,500,310,240


In [20]:
def compare_subzones_rows(df_one, df_one_column: str, df_two, df_two_column: str):
    """
    To compare the rows of 2 dataframes to check for missing rows or mismatch row names

    Parameters
    ------
    - df_one: dataframe
        the first dataframe you want to compare

    - df_one_column: str
        name of the columns of the first dataframe you want to compare
    
    - df_two: dataframe
        the second dataframe you want to compare

    - df_two_column: str
        name of the columns of the second dataframe you want to compare

    Returns
    -------
    - missing areas of the first subzone
    
    - missing areas of the second subzone
    """

    first_subzones = df_one[df_one_column].tolist()
    second_subzones = df_two[df_two_column].tolist()

    # removing all elements from the 2015 and 2020 list that contains the word total
    first_subzones = [item for item in first_subzones if "total" not in item]
    second_subzones = [item for item in second_subzones if "total" not in item]

    # removing \n
    cleaned_first_subzones = [s.replace('\n', '') for s in first_subzones]
    cleaned_second_subzones = [s.replace('\n', '') for s in second_subzones]

    missing_areas_of_first_subzone = set(cleaned_second_subzones).difference(set(cleaned_first_subzones))
    print(len(cleaned_first_subzones))
    # print(missing_areas_of_first_subzone)

    missing_areas_of_second_subzone = set(cleaned_first_subzones).difference(set(cleaned_second_subzones))
    print(len(cleaned_second_subzones))
    # print(missing_areas_of_second_subzone)

    return missing_areas_of_first_subzone, missing_areas_of_second_subzone


In [21]:
missing_areas_of_subzone_2015, missing_areas_of_subzone_2020 = compare_subzones_rows(population_by_subzone_2015, "subzone", population_by_subzone_2020, "subzone")

print(missing_areas_of_subzone_2015)
print(missing_areas_of_subzone_2020)

323
332
{'cleantech', 'garden', 'forest hill', 'lakeside (leisure)', 'bahar', 'park', 'brickland', 'choa chu kangnorth', 'murai', 'lakeside (business)', 'tengah industrial estate', 'plantation', 'nicoll'}
{'tengah', 'western water catchment', 'lakeside', 'choa chu kang north'}


#### Changes to the subzone naming between 2019 and 2014 masterplan geospatial.
western water catchment has been renamed to Murai in the 2019 for the western water catchment planning area

lakeside for jurong east has been split into lakeside (business) and lakeside (leisure)

Tengah subzone (2014) has been split into Brickland, Forest Hill, Garden, Park, Plantation, Tengah Industrial Estate (2019)

### Cleaning up population dataset
Changes to make to standardise the dataset:
* For 2020: **change 'choa chu kangnorth' to 'choa chu kang north'**


In [22]:
## decided not to remove any subzone in 2020 to match 2015,
## as the masterplan geospatial data matches the census data's lists of subzones
# subzones_to_remove_from_2020 = ["brickland", "garden", "park", "plantation",
#                       "tengah industrial estate", "bahar", "cleantech"]

subzone_name_mapping = {
    'choa chu kangnorth': 'choa chu kang north',
    # 'forest hill': 'tengah',
    # 'murai': 'western water catchment',
}

In [23]:
# Keep rows where Subzone names is NOT in the list of subzones to remove
# filtered_subzone_2020 = population_by_subzone_2020[
#     ~population_by_subzone_2020['subzone'].isin(subzones_to_remove_from_2020)
# ].copy()

filtered_subzone_2020 = population_by_subzone_2020.copy()
target_column = "subzone"
# .fillna(df[target_column]) fills in NaNs (caused by unmapped values) with the original values.
filtered_subzone_2020[target_column] = filtered_subzone_2020[target_column].map(subzone_name_mapping).fillna(filtered_subzone_2020[target_column])

missing_areas_of_subzone_2015, missing_areas_of_subzone_2020 = compare_subzones_rows(population_by_subzone_2015, "subzone", filtered_subzone_2020, "subzone")

print("subzone name missing/mismatch from 2015 population dataset: ", missing_areas_of_subzone_2015)
print("subzone name missing/mismatch from 2020 population dataset: ", missing_areas_of_subzone_2020)

323
332
subzone name missing/mismatch from 2015 population dataset:  {'cleantech', 'garden', 'forest hill', 'lakeside (leisure)', 'bahar', 'park', 'brickland', 'murai', 'lakeside (business)', 'tengah industrial estate', 'plantation', 'nicoll'}
subzone name missing/mismatch from 2020 population dataset:  {'tengah', 'western water catchment', 'lakeside'}


#### cleaning up dataset and adding total_above_60 column

In [24]:
# Handle String Representations: Replace all strings that look like missing values
filtered_subzone_2020 = (
    filtered_subzone_2020.replace(MISSING_VALUES, np.nan, regex=False))

population_by_subzone_2015 = (
    population_by_subzone_2015.replace(MISSING_VALUES, np.nan, regex=False))

convert_cols_to_int(filtered_subzone_2020)
convert_cols_to_int(population_by_subzone_2015)

cols_2015 = ['total_60_64', 'total_65_69', 'total_70_74', 'total_75_79', 'total_80_84', 'total_85andover']
cols_2020 = ['60 - 64', '65 - 69', '70 - 74', '75 - 79', '80 - 84', '85 - 89', '90 & over']

population_by_subzone_2015["total_above_60"] = population_by_subzone_2015[cols_2015].sum(axis=1)
filtered_subzone_2020["total_above_60"] = filtered_subzone_2020[cols_2020].sum(axis=1)

### Save the cleaned subzone data to the datasets/singapore_data/cleaned_data folder


In [25]:
population_2015_filepath = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/2015_age_group_planning_area_subzone.xlsx")
population_2020_filepath = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/2020_age_group_planning_area_subzone.xlsx")



save_to_excel_with_multiple_sheets(filtered_subzone_2020, population_2020_filepath, "subzone")

population_by_subzone_2015['planning_area'] = population_by_subzone_2015['planning_area'].str.replace('- total', '').str.strip()
save_to_excel_with_multiple_sheets(population_by_subzone_2015, population_2015_filepath, "subzone")

Save to filepath: /Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/singapore_data/cleaned_data/2020_age_group_planning_area_subzone.xlsx
Save to filepath: /Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/singapore_data/cleaned_data/2015_age_group_planning_area_subzone.xlsx


In [26]:
population_by_subzone_2015.columns

Index(['planning_area', 'subzone', 'total_total', 'total_0_4', 'total_5_9',
       'total_10_14', 'total_15_19', 'total_20_24', 'total_25_29',
       'total_30_34', 'total_35_39', 'total_40_44', 'total_45_49',
       'total_50_54', 'total_55_59', 'total_60_64', 'total_65_69',
       'total_70_74', 'total_75_79', 'total_80_84', 'total_85andover',
       'males_total', 'males_0_4', 'males_5_9', 'males_10_14', 'males_15_19',
       'males_20_24', 'males_25_29', 'males_30_34', 'males_35_39',
       'males_40_44', 'males_45_49', 'males_50_54', 'males_55_59',
       'males_60_64', 'males_65_69', 'males_70_74', 'males_75_79',
       'males_80_84', 'males_85andover', 'females_total', 'females_0_4',
       'females_5_9', 'females_10_14', 'females_15_19', 'females_20_24',
       'females_25_29', 'females_30_34', 'females_35_39', 'females_40_44',
       'females_45_49', 'females_50_54', 'females_55_59', 'females_60_64',
       'females_65_69', 'females_70_74', 'females_75_79', 'females_80_84',
  

#### Going through 2010 dataset to see changes in planning area and subzone comapred to 2015.

In [27]:
population_2010_filepath = Path(BASE_DATASET_PATH / "singapore_data/singstat/2010_age_group_subzone_shorten.xlsx")

population_2010_df = pd.read_excel(population_2010_filepath, sheet_name = "planning_area")

# make the column names all lower case
population_2010_df.columns = [x.lower() for x in population_2010_df.columns]

# convert all elements in the dataframe to string and lower case
population_2010_df = population_2010_df.apply(lambda x: x.astype(str).str.lower())
# removing "\n"
# population_2010_df['planning_area'] = population_2020_df['planning_area'].str.replace(r'\n', '', regex=True)

print(population_2010_df.shape)
population_2010_df.head()

(37, 16)


,planning_area,total,0 - 4,5 - 9,10 - 14,15 - 19,20 - 24,25 - 29,30 - 34,35 - 39,40 - 44,45 - 49,50 - 54,55 - 59,60 - 64,65 & over
0,total,3771721,194432,215675,244302,263750,247190,272639,298687,320024,309441,323459,303044,248696,191995,338387
1,ang mo kio,179297,7967,8424,9335,10457,10656,13400,14502,14510,13525,14862,14605,13785,11868,21401
2,bedok,294519,13230,15018,17489,20083,20156,21265,21707,22751,22018,24615,24632,21891,18018,31646
3,bishan,91298,3941,4759,5691,7188,6338,5930,6010,7354,7102,8041,8245,6725,5141,8833
4,bukit batok,144198,7187,8516,9738,10427,10625,11332,10941,11829,12259,13049,12433,9911,6487,9464


In [28]:
areas_2015 = planning_area_2015["planning_area"].tolist()
areas_2015 = [s.replace('- total', '') for s in areas_2015]
areas_2015.sort()
areas_2010 = population_2010_df["planning_area"].tolist()
areas_2010 = [s.replace('\n', '') for s in areas_2010]
index_of_total = areas_2010.index("total")
del areas_2010[index_of_total]
areas_2010.sort()
print("Number of areas listed in 2015 data:", len(areas_2015))
print(areas_2015)
print("Number of areas listed in 2010 data:", len(areas_2010))
print(areas_2010)

Number of areas listed in 2015 data: 56
['ang mo kio', 'bedok', 'bishan', 'boon lay', 'bukit batok', 'bukit merah', 'bukit panjang', 'bukit timah', 'central water catchment', 'changi', 'changi bay', 'choa chu kang', 'clementi', 'downtown core', 'geylang', 'hougang', 'jurong east', 'jurong west', 'kallang', 'lim chu kang', 'mandai', 'marina east', 'marina south', 'marine parade', 'museum', 'newton', 'north-eastern islands', 'novena', 'orchard', 'outram', 'pasir ris', 'paya lebar', 'pioneer', 'punggol', 'queenstown', 'river valley', 'rochor', 'seletar', 'sembawang', 'sengkang', 'serangoon', 'simpang', 'singapore river', 'southern islands', 'straits view', 'sungei kadut', 'tampines', 'tanglin', 'tengah', 'toa payoh', 'total', 'tuas', 'western islands', 'western water catchment', 'woodlands', 'yishun']
Number of areas listed in 2010 data: 36
['ang mo kio', 'bedok', 'bishan', 'bukit batok', 'bukit merah', 'bukit panjang', 'bukit timah', 'changi', 'choa chu kang', 'clementi', 'downtown core'

In [29]:
missing_areas_of_areas_2015 = set(areas_2010).difference(set(areas_2015))
print(missing_areas_of_areas_2015)

missing_areas_of_areas_2010 = set(areas_2015).difference(set(areas_2010))
print(missing_areas_of_areas_2010)

{'others'}
{'marina south', 'western water catchment', 'changi bay', 'simpang', 'marina east', 'museum', 'straits view', 'tengah', 'boon lay', 'lim chu kang', 'pioneer', 'western islands', 'north-eastern islands', 'total', 'tuas', 'central water catchment', 'orchard', 'sungei kadut', 'paya lebar', 'seletar', 'southern islands'}


#### The missing planning area in the 2010 dataset could be the fact that some planning area does not have population in them

In [30]:
### saving 2010 age group by planning area data
population_2010_filepath = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/2010_age_group_planning_area_subzone.xlsx")
convert_cols_to_int(population_2010_df)
cols_2010 = ['60 - 64', '65 & over']
population_2010_df["total_above_60"] = population_2010_df[cols_2010].sum(axis=1)
save_to_excel_with_multiple_sheets(population_2010_df, population_2010_filepath, "planning_area")

Save to filepath: /Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/singapore_data/cleaned_data/2010_age_group_planning_area_subzone.xlsx


In [31]:
population_2010_filepath = Path(BASE_DATASET_PATH / "singapore_data/singstat/2010_age_group_subzone_shorten.xlsx")

population_subzone_2010 = pd.read_excel(population_2010_filepath, sheet_name = "subzone")
population_by_subzone_2010 = clean_subzone_column(population_subzone_2010, "subzone")

mask_to_remove = population_by_subzone_2015['subzone'].str.contains('total', na=False)
population_by_subzone_2015 = population_by_subzone_2015[~mask_to_remove].copy()

subzone_list_2010 = population_by_subzone_2010["subzone"].tolist()
subzone_list_2015 = population_by_subzone_2015["subzone"].tolist()
print(len(subzone_list_2010))
print(subzone_list_2010)

191
['cheng san', 'chong boon', 'kebun bahru', 'sembawang hills', 'shangri-la', 'sindo', 'town centre', 'townsville', 'yio chu kang', 'bayshore', 'bedok north', 'bedok reservoir', 'bedok south', 'frankel', 'kaki bukit', 'kembangan', 'siglap', 'bishan east', 'marymount', 'upper thomson', 'bukit batok central', 'bukit batok east', 'bukit batok south', 'bukit batok west', 'gombak', 'guilin', 'hillview', 'hong kah', 'alexandra hill', 'bukit ho swee', 'bukit merah', 'depot road', 'everton park', 'henderson hill', 'kampong tiong bahru', 'maritime square', 'redhill', 'telok blangah drive', 'telok blangah rise', 'telok blangah way', 'tiong bahru', 'tiong bahru station', 'bangkit', 'bukit panjang', 'dairy farm', 'fajar', 'nature reserve', 'saujana', 'senja', 'anak bukit', 'coronation road', 'farrer court', 'hillcrest', 'holland road', 'leedon park', 'swiss club', 'ulu pandan', 'changi west', 'central', 'keat hong', 'kranji north', 'pang sua', 'peng siang', 'teck whye', 'yew tee', 'clementi cent

In [32]:
missing_areas_of_subzone_2010, missing_areas_of_subzone_2015 = compare_subzones_rows(population_by_subzone_2010, "subzone", population_by_subzone_2015, "subzone")

print(missing_areas_of_subzone_2010)
print(missing_areas_of_subzone_2015)

191
323
{'punggol town centre', 'tuas view', 'coney island', 'pasir ris wafer fab park', 'turf club', 'marina centre', 'pulau seletar', 'alexandra north', 'waterway east', 'selegie', 'straits view', 'gul circle', 'istana negara', 'tagore', 'jurong gateway', 'southern group', 'tanjong irau', 'sembawang east', 'kian teck', 'loyang west', 'tai seng', 'hougang west', 'simpang north', 'brickworks', 'gul basin', 'bayfront subzone', 'mandai estate', 'dhoby ghaut', 'city hall', 'one north', 'chin bee', 'goodwood park', 'queensway', 'paya lebar north', 'kranji', 'sentosa', 'samulun', 'pandan', 'east coast', 'western water catchment', 'lorong halus', 'liu fang', 'changi bay', 'yuhua west', 'phillip', 'international business park', "people's park", 'boon lay place', 'rochor canal', 'fernvale', 'bidadari', 'kovan', 'sudong', 'lorong 8 toa payoh', 'city terminals', 'kampong glam', 'pioneer sector', 'jelebu', 'boat quay', 'gali batu', 'mandai east', 'north-eastern islands', 'upper paya lebar', 'mand

In [33]:
# # looking for subzones in 2010 that exists as a planning area in 2015
# for subzone in subzone_list_2010:
#     for planning_area in areas_2015:
#         if subzone in planning_area:
#             print(f"{subzone} is a planning area in 2015, {planning_area}")

In [34]:
# # looking for duplicates in the 2010 subzone list
# duplicate_items = find_all_duplicates(subzone_list_2010)
# print(f"Duplicate items: {duplicate_items}")

In [35]:
# Handle String Representations: Replace all strings that look like missing values

# Create the regex pattern using word boundaries (\b)
# Word boundaries ensure that "nan" doesn't match a part of "banana"
# We escape the hyphen (-) for proper regex behavior, and join the list with '|' (OR)
pattern_to_join = '|'.join([val.replace('-', r'\-') for val in MISSING_VALUES])

# The r'\b(...)\b' ensures only whole tokens are matched
regex_pattern_fixed = r'\b(' + pattern_to_join + r')\b'

population_subzone_2010["subzone"] = (population_subzone_2010["subzone"]
                                      .replace(regex_pattern_fixed, "total", regex = True)
                                      )
# Handle True Missing Values (np.nan): Replace the standard float np.nan/None
population_subzone_2010['subzone'] = population_subzone_2010['subzone'].fillna('total')
population_subzone_2010.head()

,planning_area,subzone,total,0 - 4,5 - 9,10 - 14,15 - 19,20 - 24,25 - 29,30 - 34,35 - 39,40 - 44,45 - 49,50 - 54,55 - 59,60 - 64,65 & Over
0,Total,total,3771721,194432,215675,244302,263750,247190,272639,298687,320024,309441,323459,303044,248696,191995,338387
1,Ang Mo Kio,total,179297,7967,8424,9335,10457,10656,13400,14502,14510,13525,14862,14605,13785,11868,21401
2,NaN,cheng san,30503,1334,1387,1428,1587,1720,2471,2691,2569,2365,2471,2461,2487,2099,3433
3,NaN,chong boon,29903,1254,1304,1429,1563,1713,2348,2469,2326,2241,2431,2388,2380,2137,3920
4,NaN,kebun bahru,25854,1089,1189,1327,1418,1460,1901,2076,2123,1968,2122,2078,1992,1752,3359


In [36]:
# lower the planning area names
population_subzone_2010["planning_area"] = population_subzone_2010["planning_area"].str.strip().str.lower()

population_2010_filepath = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/2010_age_group_planning_area_subzone.xlsx")
convert_cols_to_int(population_subzone_2010)
cols_2010 = ['60 - 64', '65 & Over']
population_subzone_2010["total_above_60"] = population_subzone_2010[cols_2010].sum(axis=1)
save_to_excel_with_multiple_sheets(population_subzone_2010, population_2010_filepath, "subzone")

Save to filepath: /Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/singapore_data/cleaned_data/2010_age_group_planning_area_subzone.xlsx
